# 14) Target-Feature Bivariate Analysis (Thesis Audit)

Ziel dieses Notebooks ist die bivariate Evidenz fuer die Zielvariable
`target_afrr_activation_price_vwap_pos_h1`:

1. Top-15 Feature-Korrelationen (Pearson + Spearman) inkl. p-Werten.
2. Scatterplots mit Regressionslinie fuer die Top-5 Features.
3. Familien-Heatmaps (z. B. DA-Derivate).
4. Regime-Analyse fuer kategoriale Features (`is_weekend`,
   `TE_hour_regime_activation_lag_1h`).


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

from energy_trading.visualization.style import apply_geo_style, THESIS_PALETTE

warnings.filterwarnings('ignore')
apply_geo_style()


def resolve_repo_root() -> Path:
    root = Path.cwd().resolve()
    if (root / 'src').exists():
        return root
    for p in root.parents:
        if (p / 'src').exists():
            return p
    raise RuntimeError("Could not resolve repo root (directory containing 'src').")


REPO_ROOT = resolve_repo_root()
FEATURE_PATH = REPO_ROOT / 'data/features/all_data_features.parquet'
IMPORTANCE_PATH = REPO_ROOT / 'data/reports/model_training/importance_report.csv'
REPORT_DIR = REPO_ROOT / 'data/reports/model_training'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'target_afrr_activation_price_vwap_pos_h1'

print('REPO_ROOT:', REPO_ROOT)
print('FEATURE_PATH:', FEATURE_PATH)
print('IMPORTANCE_PATH:', IMPORTANCE_PATH)


In [ ]:
assert FEATURE_PATH.exists(), f'Missing feature artifact: {FEATURE_PATH}'

df = pd.read_parquet(FEATURE_PATH)
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True, errors='coerce')
df = df.sort_values('timestamp_utc').reset_index(drop=True)

if TARGET_COL not in df.columns:
    raise KeyError(f'Missing required target column: {TARGET_COL}')

print('Rows:', len(df), 'Cols:', len(df.columns))
print('Time span:', df['timestamp_utc'].min(), '->', df['timestamp_utc'].max())
print('Target NaNs:', int(df[TARGET_COL].isna().sum()))


In [ ]:
# Top-15 Features aus Importance-Report (Gain/SHAP).
# Falls der Report fehlt, Fallback: Top-15 nach |Pearson| (nur fuer lauffaehiges Audit).

numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
feature_candidates = [c for c in numeric_cols if c != TARGET_COL and not c.startswith('target_')]

top15_features = []
source = ''

if IMPORTANCE_PATH.exists():
    imp = pd.read_csv(IMPORTANCE_PATH)
    if 'feature' not in imp.columns:
        raise KeyError("importance_report.csv must contain column 'feature'.")

    rank_col = 'gain_rank' if 'gain_rank' in imp.columns else None
    if rank_col is None and 'xgboost_gain' in imp.columns:
        imp = imp.sort_values('xgboost_gain', ascending=False)
    elif rank_col is not None:
        imp = imp.sort_values(rank_col, ascending=True)

    top15_features = [f for f in imp['feature'].tolist() if f in df.columns][:15]
    source = 'importance_report.csv'
else:
    corr_fallback = []
    y = pd.to_numeric(df[TARGET_COL], errors='coerce')
    for c in feature_candidates:
        x = pd.to_numeric(df[c], errors='coerce')
        m = x.notna() & y.notna()
        if m.sum() < 10:
            continue
        r = np.corrcoef(x[m], y[m])[0, 1]
        if np.isfinite(r):
            corr_fallback.append((c, abs(float(r))))
    corr_fallback = sorted(corr_fallback, key=lambda t: t[1], reverse=True)
    top15_features = [c for c, _ in corr_fallback[:15]]
    source = 'fallback_abs_pearson'

if len(top15_features) < 5:
    raise ValueError('Too few usable top features for bivariate analysis.')

print('Top-15 source:', source)
print(top15_features)


In [ ]:
# Pearson + Spearman + p-Werte fuer Top-15
rows = []
y = pd.to_numeric(df[TARGET_COL], errors='coerce')

for feat in top15_features:
    x = pd.to_numeric(df[feat], errors='coerce')
    m = x.notna() & y.notna()
    n = int(m.sum())
    if n < 10:
        rows.append({
            'feature': feat,
            'n_obs': n,
            'pearson_r': np.nan,
            'pearson_p': np.nan,
            'spearman_rho': np.nan,
            'spearman_p': np.nan,
            'significant_5pct': False,
        })
        continue

    pr, pp = pearsonr(x[m], y[m])
    sr, sp = spearmanr(x[m], y[m])
    rows.append({
        'feature': feat,
        'n_obs': n,
        'pearson_r': float(pr),
        'pearson_p': float(pp),
        'spearman_rho': float(sr),
        'spearman_p': float(sp),
        'significant_5pct': bool((pp < 0.05) or (sp < 0.05)),
    })

corr_report = pd.DataFrame(rows).sort_values('spearman_rho', key=lambda s: s.abs(), ascending=False)
out_corr = REPORT_DIR / 'bivariate_top15_correlations.csv'
corr_report.to_csv(out_corr, index=False)

corr_report


In [ ]:
# Scatterplots mit Regressionslinie fuer Top-5 Features
plot_feats = corr_report['feature'].head(5).tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(plot_feats):
    ax = axes[i]
    tmp = df[[feat, TARGET_COL]].copy()
    tmp[feat] = pd.to_numeric(tmp[feat], errors='coerce')
    tmp[TARGET_COL] = pd.to_numeric(tmp[TARGET_COL], errors='coerce')
    tmp = tmp.dropna()

    if len(tmp) > 5000:
        tmp = tmp.sample(5000, random_state=42)

    sns.regplot(
        data=tmp,
        x=feat,
        y=TARGET_COL,
        scatter_kws={'alpha': 0.35, 's': 18, 'color': THESIS_PALETTE['primary']},
        line_kws={'color': THESIS_PALETTE['tertiary'], 'linewidth': 2},
        ax=ax,
    )
    ax.set_title(f'{feat} vs {TARGET_COL}')

for j in range(len(plot_feats), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
out_scatter = REPORT_DIR / 'bivariate_top5_scatter_regplot.png'
plt.savefig(out_scatter, dpi=160)
plt.show()
out_scatter


In [ ]:
# Heatmap: DA-Preis-Derivate-Familie + Target

da_family = [
    'da_price_pit',
    'da_price_slog1p',
    'da_price_diff1',
    'da_price_diff24',
    'da_price_ewma24',
    'da_price_mean_24h',
    'da_price_std_24h',
    'da_price_mean_168h',
    'da_price_std_168h',
    'da_price_volatility_30d',
    TARGET_COL,
]
da_family = [c for c in da_family if c in df.columns]

if len(da_family) >= 3:
    corr_da = df[da_family].apply(pd.to_numeric, errors='coerce').corr(method='spearman')
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_da, cmap='RdBu_r', center=0, annot=False)
    plt.title('Spearman-Heatmap: DA-Derivate-Familie + Target')
    plt.tight_layout()
    out_da = REPORT_DIR / 'heatmap_da_family_target.png'
    plt.savefig(out_da, dpi=160)
    plt.show()
    out_da
else:
    print('Zu wenige DA-Familien-Spalten fuer Heatmap gefunden.')


In [ ]:
# Heatmap: Solar-Forecast-Familie + Target
solar_family = [c for c in df.columns if c.startswith('solar_forecast')]
for c in [TARGET_COL]:
    if c in df.columns:
        solar_family.append(c)

solar_family = list(dict.fromkeys(solar_family))
if len(solar_family) >= 3:
    corr_solar = df[solar_family].apply(pd.to_numeric, errors='coerce').corr(method='spearman')
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_solar, cmap='RdBu_r', center=0, annot=False)
    plt.title('Spearman-Heatmap: Solar-Forecast-Familie + Target')
    plt.tight_layout()
    out_solar = REPORT_DIR / 'heatmap_solar_family_target.png'
    plt.savefig(out_solar, dpi=160)
    plt.show()
    out_solar
else:
    print('Zu wenige Solar-Familien-Spalten fuer Heatmap gefunden.')


In [ ]:
# Regime-Analyse: kategoriale Features vs Target-Verteilung
regime_cols = [c for c in ['is_weekend', 'TE_hour_regime_activation_lag_1h'] if c in df.columns]

if not regime_cols:
    print('Keine Regime-Spalten gefunden.')
else:
    fig, axes = plt.subplots(1, len(regime_cols), figsize=(7 * len(regime_cols), 5))
    if len(regime_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, regime_cols):
        tmp = df[[col, TARGET_COL]].copy()
        tmp[col] = pd.to_numeric(tmp[col], errors='coerce')
        tmp[TARGET_COL] = pd.to_numeric(tmp[TARGET_COL], errors='coerce')
        tmp = tmp.dropna()

        if col == 'TE_hour_regime_activation_lag_1h':
            tmp[col] = tmp[col].round().astype(int)
        else:
            tmp[col] = tmp[col].astype(int)

        sns.boxplot(data=tmp, x=col, y=TARGET_COL, ax=ax, color=THESIS_PALETTE['secondary'])
        ax.set_title(f'Target-Verteilung nach {col}')

    plt.tight_layout()
    out_regime = REPORT_DIR / 'boxplot_target_regime_features.png'
    plt.savefig(out_regime, dpi=160)
    plt.show()
    out_regime


In [ ]:
# Signifikanz-Zusammenfassung
sig = corr_report.copy()
sig['pearson_sig_5pct'] = sig['pearson_p'] < 0.05
sig['spearman_sig_5pct'] = sig['spearman_p'] < 0.05
sig['both_sig_5pct'] = sig['pearson_sig_5pct'] & sig['spearman_sig_5pct']

print('Signifikant (mind. ein Test, 5%):', int(sig['significant_5pct'].sum()), 'von', len(sig))
print('Signifikant (beide Tests, 5%):', int(sig['both_sig_5pct'].sum()), 'von', len(sig))

out_sig = REPORT_DIR / 'bivariate_significance_summary.csv'
sig.to_csv(out_sig, index=False)
out_sig


## Methodischer Hinweis

- Dieses Notebook ist **bivariat** und liefert Evidenz zur Zusammenhangsstaerke
  einzelner Merkmale mit dem Target.
- Fuer kausale Modellguete ist weiterhin die multivariate Validierung (Purged CV,
  MAE/RMSE/PnL) massgeblich.
- Bei hoher Korrelation innerhalb einzelner Familien (z. B. Solar-Forecasts)
  wird fuer XGBoost standardmaessig **keine erzwungene PCA** angewendet, da
  Baumverfahren robuster mit Kollinearitaet umgehen und Rohfeatures besser
  interpretierbar bleiben.
